# SKANN-SSL V3 Training (V2 Architecture)

**Base:** V2.1.0 architecture (proven, untouched)  
**Data:** V3 dataset (5-second clips, 5 classes)  
**Platform:** Google Colab A100

---

## Architecture: SKANN (Selective Kernel Audio Neural Networks)

| Component | Description |
|-----------|-------------|
| SKFilterbank | Multi-scale 1D convolutions with attention-weighted fusion |
| SK Kernels | **(31, 63, 127, 255, 511, 1023)** - underwater appropriate |
| Frequency coverage | 15 Hz - 500 Hz (shaft rate → cavitation) |
| 2D Backbone | Conv2d with AdaptiveAvgPool2d(1) → 512-dim **h** |
| Projector | 512→4096→8192→16384→256 → **z** |

## SSL Health Metrics (every 10 epochs)

| Metric | What it measures |
|--------|------------------|
| Silhouette on **h** | Backbone feature quality (what gets deployed) |
| Silhouette on **z** | Projector output quality |
| kNN accuracy on **h** | Classification potential of backbone |

**Diagnostic:** If sil_h > 0.5 but sil_z < 0.3 → Projector problem

## Cell 0: Mount Drive & Requirements

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q umap-learn joblib

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    
    if vram_gb >= 70:
        print("\n✅ A100 80GB detected - can use batch_size=32-48")
    elif vram_gb >= 35:
        print("\n✅ A100 40GB detected - recommended batch_size=24")
    else:
        print(f"\n⚠️ {vram_gb:.0f}GB VRAM - may need batch_size=8-16")

## Cell 1: Configuration

In [ ]:
import os
import time
from datetime import datetime

def log(msg):
    ts = time.strftime('%H:%M:%S')
    print(f"[{ts}] {msg}", flush=True)

# =============================================================================
# PATHS
# =============================================================================
DATA_ROOT = "/content/drive/MyDrive/SKANN_SSL/data/prototype_dataset"
MANIFEST_PATH = os.path.join(DATA_ROOT, "pairing_manifest.csv")
TENSOR_DIR = os.path.join(DATA_ROOT, "tensors")

OUTPUT_DIR = "/content/drive/MyDrive/SKANN-SSL/v3_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# HYPERPARAMETERS
# =============================================================================
EPOCHS = 50
BATCH_SIZE = 24
BASE_LR = 1e-4
WEIGHT_DECAY = 0.01
BT_LAMBDA = 5e-3
LATENT_DIM = 256

# Logging
LOG_EVERY = 2                # Console log every N epochs
HEALTH_CHECK_EVERY = 10      # Full health metrics every N epochs
CHECKPOINT_EVERY = 10

print("="*60)
print("SKANN-SSL V3 Configuration")
print("="*60)
print(f"Data root: {DATA_ROOT}")
print(f"Output: {OUTPUT_DIR}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Latent dim: {LATENT_DIM}")
print(f"SK Kernels: (31, 63, 127, 255, 511, 1023)")
print(f"Projector: 512 → 4096 → 8192 → 16384 → {LATENT_DIM}")
print(f"Health check every: {HEALTH_CHECK_EVERY} epochs")
print(f"\nLoss history will be saved to: {OUTPUT_DIR}/loss_history.csv")
print("="*60)

## Cell 2: Validate Data

In [ ]:
import pandas as pd
import numpy as np

assert os.path.exists(MANIFEST_PATH), f"Manifest not found: {MANIFEST_PATH}"
assert os.path.exists(TENSOR_DIR), f"Tensors not found: {TENSOR_DIR}"

manifest_df = pd.read_csv(MANIFEST_PATH)
log(f"Manifest: {len(manifest_df)} entries")

print(f"\nClass distribution:")
for cls, count in manifest_df['vessel_class'].value_counts().items():
    print(f"  {cls}: {count}")

tensor_files = [f for f in os.listdir(TENSOR_DIR) if f.endswith('.npy')]
log(f"Tensor files: {len(tensor_files)}")

sample = np.load(os.path.join(TENSOR_DIR, tensor_files[0]))
log(f"Sample shape: {sample.shape} ({sample.shape[0]/16000:.1f} sec @ 16kHz)")

## Cell 3: SKANN Model (with return_features option)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# SKANN: SELECTIVE KERNEL AUDIO NEURAL NETWORKS (V2.1.0)
# SK Kernels: (31, 63, 127, 255, 511, 1023)
# =============================================================================

def _norm_1d(channels, kind='gn', groups=8):
    if kind == 'bn':
        return nn.BatchNorm1d(channels)
    if kind == 'ln':
        return nn.GroupNorm(1, channels)
    return nn.GroupNorm(min(groups, channels), channels)


class SKConv1D(nn.Module):
    """
    Selective Kernel 1D Convolution - V2.1.0
    
    SK Kernels (31, 63, 127, 255, 511, 1023) capture:
    - k=31:   Cavitation (500+ Hz)
    - k=63:   Resonance (250+ Hz)
    - k=127:  Blade pass (125+ Hz)
    - k=255:  Generator 50Hz (62+ Hz)
    - k=511:  Generator 25Hz (31+ Hz)
    - k=1023: Shaft rate (15+ Hz)
    """
    
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        kernel_sizes: tuple = (31, 63, 127, 255, 511, 1023),
        stride: int = 1,
        reduction: int = 16,
        norm: str = 'gn',
        act: str = 'gelu',
        residual: bool = True,
        dropout: float = 0.0
    ):
        super().__init__()
        
        self.branches = nn.ModuleList()
        for k in kernel_sizes:
            pad = k // 2
            self.branches.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=k, stride=stride, 
                         padding=pad, bias=False)
            )
        
        self.n_branches = len(kernel_sizes)
        self.out_ch = out_ch
        self.kernel_sizes = kernel_sizes
        
        hidden = max(out_ch // reduction, 8)
        self.fc1 = nn.Linear(out_ch, hidden)
        self.fc2 = nn.Linear(hidden, out_ch * self.n_branches)
        
        self.norm = _norm_1d(out_ch, norm)
        self.act = nn.GELU() if act == 'gelu' else nn.ReLU(inplace=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        
        self.residual = residual
        self.match = None
        if residual and (in_ch != out_ch or stride != 1):
            self.match = nn.Conv1d(in_ch, out_ch, kernel_size=1, 
                                   stride=stride, bias=False)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = [branch(x) for branch in self.branches]
        U = torch.stack(feats, dim=1).sum(dim=1)
        s = F.adaptive_avg_pool1d(U, 1).squeeze(-1)
        z = self.fc2(F.relu(self.fc1(s), inplace=False))
        a = z.view(z.size(0), self.n_branches, self.out_ch)
        a = F.softmax(a, dim=1).unsqueeze(-1)
        feats_stacked = torch.stack(feats, dim=1)
        V = (a * feats_stacked).sum(dim=1)
        out = self.norm(V)
        out = self.act(out)
        out = self.dropout(out)
        if self.residual:
            res = x if self.match is None else self.match(x)
            out = out + res
        return out


class SKFilterbank(nn.Module):
    """Selective Kernel Filterbank - V2.1.0"""
    
    def __init__(
        self,
        out_ch: int = 64,
        kernel_sizes: tuple = (31, 63, 127, 255, 511, 1023),
        norm: str = 'gn'
    ):
        super().__init__()
        
        self.stem = SKConv1D(
            in_ch=1,
            out_ch=out_ch,
            kernel_sizes=kernel_sizes,
            stride=1,
            reduction=16,
            norm=norm,
            act='gelu',
            residual=False
        )
        self.post_norm = _norm_1d(out_ch, norm)
        
        print(f"    SKFilterbank kernels: {kernel_sizes}")
        print(f"    Frequency coverage @ 16kHz: {16000/kernel_sizes[-1]:.0f}Hz - {16000/kernel_sizes[0]:.0f}Hz")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        return self.post_norm(h)


class HybridSKEncoderV3(nn.Module):
    """
    SKANN-SSL V3 Encoder
    
    Forward returns:
    - return_features=False: z (projector output, 256-dim)
    - return_features=True:  (h, z) where h is backbone output (512-dim)
    """
    
    def __init__(self, latent_dim=256):
        super().__init__()
        
        print("\n" + "="*60)
        print("Building SKANN-SSL V3 Encoder")
        print("="*60)
        print("  Base: V2.1.0 architecture")
        
        # SK Frontend (V2.1.0 - UNCHANGED)
        self.sk_frontend = SKFilterbank(
            out_ch=64,
            kernel_sizes=(31, 63, 127, 255, 511, 1023)  # Underwater-appropriate!
        )
        self.downsample = nn.AvgPool1d(kernel_size=8, stride=8)
        
        # Channel bridge (V2.1.0 - UNCHANGED)
        self.channel_bridge = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=False)
        )
        
        # 2D Backbone (V2.1.0 - UNCHANGED)
        # Output: h (512-dim) - THIS IS WHAT GETS DEPLOYED
        self.backbone2d = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=False),
            nn.Conv2d(64, 128, 3, padding=1, stride=(2, 2)),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=False),
            nn.Conv2d(128, 256, 3, padding=1, stride=(2, 1)),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=False),
            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=False),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(1)
        )
        
        # Projector: 512 → 4096 → 8192 → 16384 → 256
        # Output: z (256-dim) - DISCARDED AFTER TRAINING
        self.projector = nn.Sequential(
            nn.Linear(512, 4096),
            nn.LayerNorm(4096),
            nn.ReLU(inplace=False),
            nn.Linear(4096, 8192),
            nn.LayerNorm(8192),
            nn.ReLU(inplace=False),
            nn.Linear(8192, 16384),
            nn.LayerNorm(16384),
            nn.ReLU(inplace=False),
            nn.Linear(16384, latent_dim)
        )
        
        print(f"    Backbone output (h): 512-dim")
        print(f"    Projector: 512 → 4096 → 8192 → 16384 → {latent_dim}")
        self._count_params()
        print("="*60 + "\n")
    
    def _count_params(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"    Total params: {total/1e6:.1f}M, Trainable: {trainable/1e6:.1f}M")
    
    def forward(self, x, return_features=False):
        # Shape handling
        if x.dim() > 3:
            x = x.view(x.size(0), -1).unsqueeze(1)
        elif x.dim() == 2:
            x = x.unsqueeze(1)
        
        # SK Frontend
        x = self.sk_frontend(x)
        x = self.downsample(x)
        
        # Bridge to 2D
        x = self.channel_bridge(x)
        x = x.unsqueeze(1)
        
        # 2D Backbone → h (512-dim)
        h = self.backbone2d(x)
        
        # Projector → z (256-dim)
        z = self.projector(h)
        
        if return_features:
            return h, z
        return z


# Test model
model = HybridSKEncoderV3(latent_dim=LATENT_DIM)
dummy = torch.randn(2, 1, 80000)
h, z = model(dummy, return_features=True)
print(f"✅ Test: Input {dummy.shape} → h {h.shape}, z {z.shape}")

## Cell 4: Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class HierarchicalDataset(Dataset):
    def __init__(self, manifest_path, data_dir):
        self.df = pd.read_csv(manifest_path)
        self.data_dir = data_dir
        self.class_to_id = {c: i for i, c in enumerate(sorted(self.df["vessel_class"].unique()))}
        print(f"    Classes: {list(self.class_to_id.keys())}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        anchor_id = int(row['anchor_clip_id'])
        p_ids = str(row['partner_clip_ids']).split('|')
        p_ids = [int(x) for x in p_ids if x.strip()]
        partner_id = np.random.choice(p_ids)
        
        y1 = np.load(os.path.join(self.data_dir, f"tensor_{anchor_id:06d}.npy")).flatten()
        y2 = np.load(os.path.join(self.data_dir, f"tensor_{partner_id:06d}.npy")).flatten()
        
        return (
            torch.from_numpy(y1).float(),
            torch.from_numpy(y2).float(),
            self.class_to_id[row["vessel_class"]]
        )


dataset = HierarchicalDataset(MANIFEST_PATH, TENSOR_DIR)
log(f"Dataset: {len(dataset)} samples")

## Cell 5: Barlow Twins Loss & Health Metrics

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score


def barlow_twins_loss(z1, z2, lambd=BT_LAMBDA):
    """Barlow Twins loss - V2.1.0 (unchanged)"""
    batch_size = z1.size(0)
    
    z1_mean = z1.mean(dim=0)
    z1_std = z1.std(dim=0) + 1e-6
    z1_norm = (z1 - z1_mean) / z1_std
    
    z2_mean = z2.mean(dim=0)
    z2_std = z2.std(dim=0) + 1e-6
    z2_norm = (z2 - z2_mean) / z2_std
    
    c = torch.mm(z1_norm.T, z2_norm) / batch_size
    
    diag = torch.diagonal(c)
    on_diag_loss = torch.pow(1.0 - diag, 2).sum()
    c_squared = torch.pow(c, 2)
    off_diag_loss = c_squared.sum() - torch.pow(diag, 2).sum()
    
    return on_diag_loss + lambd * off_diag_loss


def create_eval_subset(manifest_path, n_per_class=200):
    """Create balanced subset for evaluation."""
    df = pd.read_csv(manifest_path)
    classes = sorted(df['vessel_class'].unique())
    eval_indices = []
    for cls in classes:
        cls_df = df[df['vessel_class'] == cls]
        sampled = cls_df.sample(n=min(n_per_class, len(cls_df)), random_state=42)
        eval_indices.extend(sampled.index.tolist())
    return eval_indices


@torch.no_grad()
def compute_health_metrics(model, eval_indices, manifest_df, tensor_dir, class_to_id, device):
    """
    SSL Health Metrics:
    - Silhouette on h (backbone features) - DEPLOYMENT METRIC
    - Silhouette on z (projector output)
    - kNN accuracy on h (5-fold CV)
    
    Diagnostic: If sil_h > 0.5 but sil_z < 0.3 → Projector problem
    """
    model.eval()
    
    h_embeddings, z_embeddings, labels = [], [], []
    
    for idx in eval_indices:
        row = manifest_df.iloc[idx]
        clip_id = int(row['anchor_clip_id'])
        label = class_to_id[row['vessel_class']]
        
        tensor_path = os.path.join(tensor_dir, f"tensor_{clip_id:06d}.npy")
        x = np.load(tensor_path).astype(np.float32).flatten()
        x = torch.from_numpy(x).unsqueeze(0).to(device)
        
        h, z = model(x, return_features=True)
        h_embeddings.append(h.cpu().numpy().squeeze())
        z_embeddings.append(z.cpu().numpy().squeeze())
        labels.append(label)
    
    h_embeddings = np.array(h_embeddings)
    z_embeddings = np.array(z_embeddings)
    labels = np.array(labels)
    
    # Silhouette scores
    sil_h = silhouette_score(h_embeddings, labels, metric='cosine')
    sil_z = silhouette_score(z_embeddings, labels, metric='cosine')
    
    # Per-class silhouette on h
    sil_h_samples = silhouette_samples(h_embeddings, labels, metric='cosine')
    class_names = sorted(class_to_id.keys())
    per_class_h = {}
    for i, cls in enumerate(class_names):
        mask = labels == i
        if mask.sum() > 0:
            per_class_h[cls] = float(sil_h_samples[mask].mean())
    
    # kNN accuracy on h (5-fold CV)
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn_scores = cross_val_score(knn, h_embeddings, labels, cv=5, scoring='accuracy')
    knn_acc = knn_scores.mean()
    
    model.train()
    
    return {
        'sil_h': sil_h,
        'sil_z': sil_z,
        'knn_acc': knn_acc,
        'per_class_h': per_class_h
    }

## Cell 6: Training Loop with Health Monitoring

In [ ]:
from tqdm.notebook import tqdm
import glob


def train():
    log("Starting SKANN-SSL V3 Training")
    print("\n" + "="*60)
    print("SKANN-SSL V3 Training")
    print("="*60)
    print(f"  SK Kernels: (31, 63, 127, 255, 511, 1023)")
    print(f"  Projector: 512 → 4096 → 8192 → 16384 → {LATENT_DIM}")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Health check every: {HEALTH_CHECK_EVERY} epochs")
    print("="*60 + "\n")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    log(f"Device: {device}")
    
    # Model
    model = HybridSKEncoderV3(latent_dim=LATENT_DIM).to(device)
    
    # Dataset
    dataset = HierarchicalDataset(MANIFEST_PATH, TENSOR_DIR)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True
    )
    log(f"Batches per epoch: {len(loader)}")
    
    # Evaluation setup
    manifest_df = pd.read_csv(MANIFEST_PATH)
    class_to_id = {c: i for i, c in enumerate(sorted(manifest_df['vessel_class'].unique()))}
    class_names = sorted(class_to_id.keys())
    eval_indices = create_eval_subset(MANIFEST_PATH, n_per_class=200)
    log(f"Eval subset: {len(eval_indices)} clips")
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # CSV logging with health metrics
    loss_csv = os.path.join(OUTPUT_DIR, 'loss_history.csv')
    header = "epoch,loss,lr,sil_h,sil_z,knn_acc," + ",".join([f"sil_h_{c}" for c in class_names])
    with open(loss_csv, 'w') as f:
        f.write(header + "\n")
    log(f"Loss history will be saved to: {loss_csv}")
    
    # Training state
    best_sil_h = -1.0
    start_time = time.time()
    
    log(f"Starting training...\n")
    
    for epoch in tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch"):
        model.train()
        total_loss = 0.0
        
        for y1, y2, _ in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
            y1 = y1.to(device, non_blocking=True)
            y2 = y2.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            z1 = model(y1)  # Only need z for BT loss
            z2 = model(y2)
            loss = barlow_twins_loss(z1, z2)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        scheduler.step()
        avg_loss = total_loss / len(loader)
        current_lr = scheduler.get_last_lr()[0]
        
        # =================================================================
        # Health metrics (every HEALTH_CHECK_EVERY epochs)
        # =================================================================
        metrics = None
        if epoch % HEALTH_CHECK_EVERY == 0:
            metrics = compute_health_metrics(
                model, eval_indices, manifest_df, TENSOR_DIR, class_to_id, device
            )
            
            # Save best model based on backbone silhouette (h)
            if metrics['sil_h'] > best_sil_h:
                best_sil_h = metrics['sil_h']
                best_path = os.path.join(OUTPUT_DIR, "best_model.pth")
                torch.save(model.state_dict(), best_path)
                tqdm.write(f"  🏆 New best sil_h: {metrics['sil_h']:.4f}")
            
            # Diagnostic check: projector problem detection
            if metrics['sil_h'] > 0.5 and metrics['sil_z'] < 0.3:
                tqdm.write(f"  ⚠️ PROJECTOR ISSUE: sil_h={metrics['sil_h']:.4f} but sil_z={metrics['sil_z']:.4f}")
        
        # =================================================================
        # Logging (every LOG_EVERY epochs or when we have metrics)
        # =================================================================
        if epoch % LOG_EVERY == 0 or metrics is not None:
            elapsed = time.time() - start_time
            eta = elapsed / epoch * (EPOCHS - epoch)
            
            if metrics:
                # Full log with health metrics
                tqdm.write(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {current_lr:.2e}")
                tqdm.write(f"  Health: sil_h={metrics['sil_h']:.4f}, sil_z={metrics['sil_z']:.4f}, kNN={metrics['knn_acc']:.4f}")
                breakdown = " | ".join([f"{k[:4]}:{v:.3f}" for k, v in metrics['per_class_h'].items()])
                tqdm.write(f"  Per-class (h): {breakdown}")
                tqdm.write(f"  ETA: {eta/60:.1f}m")
                
                # CSV log with full metrics
                per_class_str = ",".join([f"{metrics['per_class_h'].get(c, 0):.4f}" for c in class_names])
                with open(loss_csv, 'a') as f:
                    f.write(f"{epoch},{avg_loss:.6f},{current_lr:.2e},{metrics['sil_h']:.4f},{metrics['sil_z']:.4f},{metrics['knn_acc']:.4f},{per_class_str}\n")
                    f.flush()
            else:
                # Brief log (no metrics)
                tqdm.write(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {current_lr:.2e} | ETA: {eta/60:.1f}m")
                with open(loss_csv, 'a') as f:
                    f.write(f"{epoch},{avg_loss:.6f},{current_lr:.2e},,,," + ","*(len(class_names)-1) + "\n")
        
        # =================================================================
        # Checkpoint
        # =================================================================
        if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS:
            ckpt_path = os.path.join(OUTPUT_DIR, f"BT_ckpt_epoch_{epoch:03d}.pth")
            torch.save({
                "epoch": epoch,
                "encoder": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "loss": avg_loss,
                "best_sil_h": best_sil_h,
            }, ckpt_path)
            tqdm.write(f"  💾 Checkpoint: {ckpt_path}")
    
    # Final save
    final_path = os.path.join(OUTPUT_DIR, "SKANN_SSL_V3_Final.pth")
    torch.save(model.state_dict(), final_path)
    
    total_time = time.time() - start_time
    
    print("\n" + "="*60)
    print("Training Complete")
    print("="*60)
    print(f"  Final loss: {avg_loss:.4f}")
    print(f"  Best sil_h: {best_sil_h:.4f}")
    print(f"  Total time: {total_time/60:.1f} minutes")
    print(f"  Final model: {final_path}")
    print(f"  Loss history: {loss_csv}")
    print("="*60)
    
    return model


model = train()

## Cell 7: Extract Embeddings (both h and z)

In [ ]:
@torch.no_grad()
def extract_embeddings():
    log("Extracting embeddings (h and z)...")
    
    device = torch.device('cuda')
    
    # Use model from training if available
    try:
        m = model
        log("Using model from training")
    except NameError:
        weights = os.path.join(OUTPUT_DIR, "SKANN_SSL_V3_Final.pth")
        if not os.path.exists(weights):
            cands = sorted(glob.glob(os.path.join(OUTPUT_DIR, "BT_ckpt_epoch_*.pth")))
            if not cands:
                log("❌ No weights found!")
                return None, None, None
            weights = cands[-1]
        
        m = HybridSKEncoderV3(latent_dim=LATENT_DIM).to(device)
        state = torch.load(weights, map_location=device)
        if 'encoder' in state:
            state = state['encoder']
        m.load_state_dict(state)
        log(f"Loaded: {weights}")
    
    m.eval()
    
    files = sorted(glob.glob(os.path.join(TENSOR_DIR, 'tensor_*.npy')))
    
    manifest = pd.read_csv(MANIFEST_PATH)
    clip_to_class = dict(zip(manifest['anchor_clip_id'], manifest['vessel_class']))
    class_to_id = {c: i for i, c in enumerate(sorted(manifest['vessel_class'].unique()))}
    
    h_embeddings, z_embeddings, labels = [], [], []
    
    for f in tqdm(files, desc="Extracting"):
        clip_id = int(os.path.basename(f).replace('tensor_', '').replace('.npy', ''))
        x = torch.from_numpy(np.load(f).flatten()).float().unsqueeze(0).to(device)
        
        h, z = m(x, return_features=True)
        h_embeddings.append(h.cpu().numpy().flatten())
        z_embeddings.append(z.cpu().numpy().flatten())
        labels.append(class_to_id.get(clip_to_class.get(clip_id, ''), -1))
    
    h_embeddings = np.array(h_embeddings)
    z_embeddings = np.array(z_embeddings)
    labels = np.array(labels)
    
    log(f"✅ Extracted: h={h_embeddings.shape}, z={z_embeddings.shape}")
    
    # Save
    np.save(os.path.join(OUTPUT_DIR, 'embeddings_h.npy'), h_embeddings)
    np.save(os.path.join(OUTPUT_DIR, 'embeddings_z.npy'), z_embeddings)
    np.save(os.path.join(OUTPUT_DIR, 'labels.npy'), labels)
    
    return h_embeddings, z_embeddings, labels


h_embeddings, z_embeddings, labels = extract_embeddings()

## Cell 8: Final Silhouette Scores

In [ ]:
if h_embeddings is not None:
    mask = labels >= 0
    
    # Silhouette on h (backbone) - THIS IS WHAT MATTERS
    sil_h = silhouette_score(h_embeddings[mask], labels[mask], metric='cosine')
    sil_h_samples = silhouette_samples(h_embeddings[mask], labels[mask], metric='cosine')
    
    # Silhouette on z (projector)
    sil_z = silhouette_score(z_embeddings[mask], labels[mask], metric='cosine')
    
    # kNN on h
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn_scores = cross_val_score(knn, h_embeddings[mask], labels[mask], cv=5, scoring='accuracy')
    knn_acc = knn_scores.mean()
    
    print("\n" + "="*60)
    print("FINAL METRICS")
    print("="*60)
    print(f"  Silhouette on h (backbone): {sil_h:.4f}  ← DEPLOYMENT METRIC")
    print(f"  Silhouette on z (projector): {sil_z:.4f}")
    print(f"  kNN accuracy on h: {knn_acc:.4f}")
    print("="*60)
    
    # Diagnostic
    if sil_h > 0.5 and sil_z < 0.3:
        print("\n⚠️ PROJECTOR ISSUE DETECTED")
        print(f"   Backbone is good (sil_h={sil_h:.4f})")
        print(f"   But projector collapsed (sil_z={sil_z:.4f})")
    
    print(f"\nComparison (backbone silhouette):")
    print(f"  V1 baseline: 0.3997")
    print(f"  V2.1.0:      0.8299")
    print(f"  V3:          {sil_h:.4f}")
    
    # Per-class on h
    manifest = pd.read_csv(MANIFEST_PATH)
    classes = sorted(manifest['vessel_class'].unique())
    
    print(f"\nPer-class silhouette (h):")
    for i, cls in enumerate(classes):
        cls_mask = labels[mask] == i
        if cls_mask.sum() > 0:
            cls_sil = sil_h_samples[cls_mask].mean()
            status = "✅" if cls_sil > 0.5 else "⚠️" if cls_sil > 0 else "❌"
            print(f"  {cls}: {cls_sil:.4f} {status}")
    
    # Save
    with open(os.path.join(OUTPUT_DIR, 'final_metrics.txt'), 'w') as f:
        f.write(f"sil_h: {sil_h:.6f}\n")
        f.write(f"sil_z: {sil_z:.6f}\n")
        f.write(f"knn_acc: {knn_acc:.6f}\n")
        f.write(f"\nPer-class h:\n")
        for i, cls in enumerate(classes):
            cls_mask = labels[mask] == i
            if cls_mask.sum() > 0:
                f.write(f"{cls}: {sil_h_samples[cls_mask].mean():.6f}\n")

## Cell 9: t-SNE Visualization

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

if h_embeddings is not None:
    log("Generating t-SNE (on backbone features h)...")
    
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    h_2d_tsne = tsne.fit_transform(h_embeddings)
    
    manifest = pd.read_csv(MANIFEST_PATH)
    class_names = sorted(manifest['vessel_class'].unique())
    
    colors = {
        'cargo_ship': '#E31A1C',
        'fishing_vessel': '#FF7F00',
        'small_craft': '#33A02C',
        'tanker': '#6A3D9A',
        'no_vessel': '#1F78B4'
    }
    
    plt.figure(figsize=(12, 8))
    for i, cls in enumerate(class_names):
        mask = labels == i
        plt.scatter(h_2d_tsne[mask, 0], h_2d_tsne[mask, 1],
                   c=colors.get(cls, 'gray'), label=cls,
                   s=40, alpha=0.7, edgecolors='white', linewidth=0.5)
    
    plt.title(f"SKANN-SSL V3: t-SNE of Backbone Features (h)\nSilhouette(h): {sil_h:.4f} | kNN: {knn_acc:.4f}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    
    tsne_path = os.path.join(OUTPUT_DIR, 'tsne_h_v3.png')
    plt.savefig(tsne_path, dpi=300, bbox_inches='tight')
    plt.show()
    log(f"✅ t-SNE saved: {tsne_path}")

## Cell 10: UMAP Visualization

In [ ]:
import umap

if h_embeddings is not None:
    log("Generating UMAP (on backbone features h)...")
    
    umap_config = {'n_neighbors': 15, 'min_dist': 0.1, 'metric': 'cosine', 'random_state': 42}
    reducer = umap.UMAP(**umap_config)
    h_2d_umap = reducer.fit_transform(h_embeddings)
    
    plt.figure(figsize=(12, 8))
    for i, cls in enumerate(class_names):
        mask = labels == i
        plt.scatter(h_2d_umap[mask, 0], h_2d_umap[mask, 1],
                   c=colors.get(cls, 'gray'), label=cls,
                   s=40, alpha=0.7, edgecolors='white', linewidth=0.5)
    
    plt.title(f"SKANN-SSL V3: UMAP of Backbone Features (h)\nSilhouette(h): {sil_h:.4f} | kNN: {knn_acc:.4f}")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    
    umap_path = os.path.join(OUTPUT_DIR, 'umap_h_v3.png')
    plt.savefig(umap_path, dpi=300, bbox_inches='tight')
    plt.show()
    log(f"✅ UMAP saved: {umap_path}")

## Cell 11: Territory Mapping

In [ ]:
from sklearn.metrics import pairwise_distances
from scipy.spatial import Voronoi
from matplotlib.patches import Polygon
import joblib

if h_embeddings is not None and h_2d_umap is not None:
    log("Computing territory mapping...")
    
    manifest = pd.read_csv(MANIFEST_PATH)
    vessel_classes = sorted(manifest['vessel_class'].unique())
    class_to_idx = {c: i for i, c in enumerate(vessel_classes)}
    
    # Compute centroids on h (backbone) - not z!
    centroids = {}
    centroid_stats = {}
    centroids_2d = {}
    
    for class_name in vessel_classes:
        class_idx = class_to_idx[class_name]
        mask = labels == class_idx
        class_embeddings = h_embeddings[mask]  # Use h!
        
        centroid = class_embeddings.mean(axis=0)
        centroids[class_name] = centroid
        centroids_2d[class_name] = h_2d_umap[mask].mean(axis=0)
        
        distances_to_centroid = np.linalg.norm(class_embeddings - centroid, axis=1)
        
        centroid_stats[class_name] = {
            'n_samples': int(mask.sum()),
            'mean_distance': float(distances_to_centroid.mean()),
            'radius_95': float(np.percentile(distances_to_centroid, 95)),
        }
    
    print("\nClass centroids (h):")
    for cls in vessel_classes:
        stats = centroid_stats[cls]
        print(f"  {cls}: {stats['n_samples']} samples, 95% radius: {stats['radius_95']:.4f}")
    
    # Inter-class distances on h
    centroid_matrix = np.array([centroids[c] for c in vessel_classes])
    inter_class_dist = pairwise_distances(centroid_matrix, metric='cosine')
    
    print("\nInter-class cosine distances (h):")
    for i, c1 in enumerate(vessel_classes):
        for j, c2 in enumerate(vessel_classes):
            if i < j:
                print(f"  {c1} ↔ {c2}: {inter_class_dist[i,j]:.4f}")
    
    # Voronoi plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Plot 1: Embeddings with centroids
    ax1 = axes[0]
    for class_name in vessel_classes:
        mask = labels == class_to_idx[class_name]
        ax1.scatter(h_2d_umap[mask, 0], h_2d_umap[mask, 1], 
                   c=colors.get(class_name, 'gray'), label=class_name, 
                   s=30, alpha=0.6, edgecolors='white', linewidth=0.3)
    
    for class_name in vessel_classes:
        cx, cy = centroids_2d[class_name]
        ax1.scatter(cx, cy, c=colors.get(class_name, 'gray'), s=400, marker='*', 
                   edgecolors='black', linewidth=2, zorder=10)
    
    ax1.set_xlabel('UMAP 1')
    ax1.set_ylabel('UMAP 2')
    ax1.set_title(f'V3 Backbone Embeddings (h) | Sil: {sil_h:.4f}')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Voronoi
    ax2 = axes[1]
    centroid_points = np.array([centroids_2d[c] for c in vessel_classes])
    x_min, x_max = h_2d_umap[:, 0].min() - 2, h_2d_umap[:, 0].max() + 2
    y_min, y_max = h_2d_umap[:, 1].min() - 2, h_2d_umap[:, 1].max() + 2
    
    dummy_points = np.array([[x_min-50, y_min-50], [x_max+50, y_min-50],
                             [x_min-50, y_max+50], [x_max+50, y_max+50]])
    all_points = np.vstack([centroid_points, dummy_points])
    
    try:
        vor = Voronoi(all_points)
        for idx, region_idx in enumerate(vor.point_region[:len(vessel_classes)]):
            region = vor.regions[region_idx]
            if -1 not in region and len(region) > 0:
                polygon = [vor.vertices[i] for i in region]
                poly = Polygon(polygon, facecolor=colors.get(vessel_classes[idx], 'gray'), 
                              alpha=0.3, edgecolor='black', linewidth=1.5)
                ax2.add_patch(poly)
    except Exception as e:
        print(f"  Voronoi failed: {e}")
    
    for class_name in vessel_classes:
        mask = labels == class_to_idx[class_name]
        ax2.scatter(h_2d_umap[mask, 0], h_2d_umap[mask, 1], 
                   c=colors.get(class_name, 'gray'), s=20, alpha=0.7)
    
    for class_name in vessel_classes:
        cx, cy = centroids_2d[class_name]
        ax2.scatter(cx, cy, c=colors.get(class_name, 'gray'), s=300, marker='*', 
                   edgecolors='black', linewidth=2, zorder=10)
    
    ax2.set_xlim(x_min, x_max)
    ax2.set_ylim(y_min, y_max)
    ax2.set_xlabel('UMAP 1')
    ax2.set_ylabel('UMAP 2')
    ax2.set_title('Territory Map (Voronoi)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    territory_path = os.path.join(OUTPUT_DIR, 'territory_map_v3.png')
    plt.savefig(territory_path, dpi=300, bbox_inches='tight')
    plt.show()
    log(f"✅ Territory map saved: {territory_path}")
    
    # Export bundle
    territory_bundle = {
        'centroids': {k: v.tolist() for k, v in centroids.items()},
        'centroid_stats': centroid_stats,
        'thresholds': {k: v['radius_95'] for k, v in centroid_stats.items()},
        'inter_class_distances': inter_class_dist.tolist(),
        'vessel_classes': vessel_classes,
        'metadata': {
            'version': 'v3.0.0',
            'sil_h': float(sil_h),
            'sil_z': float(sil_z),
            'knn_acc': float(knn_acc),
            'embedding_dim_h': h_embeddings.shape[1],
            'n_classes': len(vessel_classes),
            'export_date': datetime.now().isoformat(),
        }
    }
    
    bundle_path = os.path.join(OUTPUT_DIR, 'vessel_territories_v3.joblib')
    joblib.dump(territory_bundle, bundle_path, compress=3)
    log(f"✅ Territory bundle saved: {bundle_path}")

## Cell 12: Export Production Bundle

In [ ]:
def export_bundle():
    log("Exporting production bundle...")
    
    weights = os.path.join(OUTPUT_DIR, "SKANN_SSL_V3_Final.pth")
    if not os.path.exists(weights):
        weights = os.path.join(OUTPUT_DIR, "best_model.pth")
    
    state = torch.load(weights, map_location='cpu')
    if 'encoder' in state:
        state = state['encoder']
    state = {k.replace('module.', ''): v for k, v in state.items()}
    
    manifest = pd.read_csv(MANIFEST_PATH)
    vessel_labels = sorted(manifest['vessel_class'].unique())
    
    bundle = {
        'model_state': state,
        'embeddings_h': h_embeddings,
        'embeddings_z': z_embeddings,
        'labels': labels,
        'vessel_labels': vessel_labels,
        'class_map': {l: i for i, l in enumerate(vessel_labels)},
        'metrics': {
            'sil_h': float(sil_h),
            'sil_z': float(sil_z),
            'knn_acc': float(knn_acc),
            'v2_baseline': 0.8299,
            'n_samples': len(h_embeddings),
        },
        'metadata': {
            'version': 'v3.0.0',
            'architecture': 'HybridSKEncoderV3 (SKANN)',
            'sk_kernels': (31, 63, 127, 255, 511, 1023),
            'projector': '512→4096→8192→16384→256',
            'latent_dim': LATENT_DIM,
            'backbone_dim': 512,
            'clip_duration': '5 seconds',
            'n_classes': 5,
            'export_date': datetime.now().isoformat(),
        }
    }
    
    path = os.path.join(OUTPUT_DIR, 'SKANN_SSL_V3_Production_Bundle.joblib')
    joblib.dump(bundle, path, compress=3)
    
    print("\n" + "="*60)
    print("Production Bundle V3")
    print("="*60)
    print(f"  SK kernels: {bundle['metadata']['sk_kernels']}")
    print(f"  Projector: {bundle['metadata']['projector']}")
    print(f"  Backbone dim: {bundle['metadata']['backbone_dim']}")
    print(f"  Silhouette (h): {bundle['metrics']['sil_h']:.4f}  ← DEPLOYMENT METRIC")
    print(f"  Silhouette (z): {bundle['metrics']['sil_z']:.4f}")
    print(f"  kNN accuracy: {bundle['metrics']['knn_acc']:.4f}")
    print(f"  Size: {os.path.getsize(path)/1e6:.1f} MB")
    print("="*60)


export_bundle()

## Cell 13: Summary

In [ ]:
print("\n" + "="*60)
print("SKANN-SSL V3 Training Complete")
print("="*60)

print("\nArchitecture (SKANN):")
print("  SK kernels: (31, 63, 127, 255, 511, 1023)")
print("  Backbone output (h): 512-dim")
print(f"  Projector: 512 → 4096 → 8192 → 16384 → {LATENT_DIM}")

print("\nData:")
print("  Clips: 5 seconds (80,000 samples @ 16kHz)")
print("  Classes: 5")

print(f"\nFinal Metrics:")
print(f"  Silhouette (h): {sil_h:.4f}  ← DEPLOYMENT METRIC")
print(f"  Silhouette (z): {sil_z:.4f}")
print(f"  kNN accuracy: {knn_acc:.4f}")
print(f"  V2 baseline: 0.8299")

print(f"\nOutputs: {OUTPUT_DIR}")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        print(f"  {f}: {os.path.getsize(fpath)/1e6:.2f} MB")

print("\n" + "="*60)